### Stored Procedures

In [0]:

-- CREATE OR REPLACE PROCEDURE my_proc(
--     IN  p_customer_id NUMERIC,           -- p_ prefix back (avoids column clash)
--     OUT p_result TEXT
-- )
--     LANGUAGE plpgsql                          -- LANGUAGE, not "using"
-- AS $$
-- DECLARE
--     v_wallet NUMERIC;
-- BEGIN
--     SELECT wallet INTO v_wallet
--     FROM customers
--     WHERE id = p_customer_id;             -- ; ✅
--
--     IF v_wallet IS NULL THEN              -- ghost-customer check is back
--         RAISE EXCEPTION 'Customer % not found', p_customer_id;
--     ELSIF v_wallet > 1000 THEN            -- ELSIF, no E in middle
--         p_result := 'Elite Customer';     -- := and single quotes and ;
--     ELSIF v_wallet > 500 THEN
--         p_result := 'Premium Customer';
--     ELSE
--         p_result := 'Normal Customer';
--     END IF;                               -- ;
-- END;                                      -- ;
-- $$;                                       -- ;
--
-- -- CALL my_proc(1, NULL);    -- Sai, 5000  → Elite Customer
-- -- CALL my_proc(2, NULL);    -- Ravi, 500  → ?
-- -- CALL my_proc(3, NULL);    -- Priya, 0   → Normal Customer
-- -- CALL my_proc(99, NULL);   -- 💣 exception
--



-- CREATE OR REPLACE PROCEDURE print_customer_records()
--     LANGUAGE plpgsql
-- AS $$
-- DECLARE
--     rec RECORD;
-- BEGIN
--     FOR rec IN SELECT id, name FROM customers LOOP
--             RAISE NOTICE 'customer id: %  name: %', rec.id, rec.name;
--         END LOOP;
-- END;
-- $$;
-- CALL print_customer_records();



-- Question 1 — refund_order
--
-- refund_order(IN p_order_id INT)
--
-- SELECT the order's status, amount, and customer_id INTO three variables (one SELECT INTO, three columns → three variables)
-- If the order doesn't exist → RAISE EXCEPTION with the order id in the message
--     If status = 'CONFIRMED' → three statements:
-- UPDATE orders: set status to 'REFUNDED' for that order
-- UPDATE customers: wallet = wallet + the refund amount
-- INSERT into audit_log: message like 'Refund processed for order X'
--     Any other status → INSERT into audit_log: 'Refund rejected for order X'



-- create or replace procedure refund_order(
--     IN p_order_id INT
-- )
-- LANGUAGE plpgsql
-- AS $$
--     DECLARE
--         v_customer_id numeric;
--         v_amount numeric;
--         v_status TEXT;
--         st TEXT;
--     BEGIN
--         select customer_id,amount,status into v_customer_id,v_amount,v_status from orders
--         where id = p_order_id;
--
--
--         IF v_customer_id is NULL THEN
--             RAISE EXCEPTION 'order % does not exists',p_order_id;
--         ELSEIF v_status = 'CONFIRMED' THEN
--                 UPDATE orders SET status = 'REFUNDED' where id = p_order_id;
--                 UPDATE customers SET wallet = wallet + v_amount  where id = v_customer_id;
--                 st := 'Refund Processed for order ' || p_order_id;
--                 INSERT INTO audit_log (message) values (st);
--         ELSE
--             st := 'Refund Rejected for order ' || p_order_id;
--             INSERT INTO audit_log (message) values (st);
--         end if;
--     END;
-- $$;
--
--
--
--
-- -- CALL refund_order(1);    -- CONFIRMED, 800 → should refund
-- -- CALL refund_order(2);    -- PENDING → should reject
-- -- CALL refund_order(99);   -- 💥 exception
--
--
-- SELECT * FROM orders;      -- order 1 should say REFUNDED, order 2 still PENDING
-- SELECT * FROM customers;   -- Sai's wallet: 5000 + 800 = 5800
-- SELECT * FROM audit_log;   -- 'Refund Processed for order 1' AND 'Refund Rejected for order 2', with timestamps



-- create or replace procedure give_yearly_bonus()
-- language plpgsql
-- as $$
--    DECLARE
--        rec RECORD;
--        v_total_bonus NUMERIC := 0;
--        total_wallet_per_customer NUMERIC := 0;
--
--    BEGIN
--        FOR rec IN select * from customers LOOP
--            IF rec.wallet > 1000 THEN
--                UPDATE customers set wallet = wallet + (wallet/10) where id = rec.id;
--                v_total_bonus = v_total_bonus + (rec.wallet/10);
--                total_wallet_per_customer = rec.wallet + rec.wallet/10;
--            ELSE
--                UPDATE customers set wallet = wallet + 100 where id = rec.id;
--                v_total_bonus =v_total_bonus + 100;
--                total_wallet_per_customer = rec.wallet + 100;
--            end if;
--            RAISE NOTICE 'customer % and updated wallet %',rec.name,ROUND(total_wallet_per_customer,2);
--            total_wallet_per_customer = 0;
--        END LOOP;
--
--        RAISE NOTICE 'Total Bonus Paid to Employees: %',ROUND(v_total_bonus,2);
--    END;
-- $$;
--
--
-- CALL give_yearly_bonus();
--
--
-- UPDATE customers SET wallet = 5000 WHERE name = 'Sai';
-- UPDATE customers SET wallet = 500  WHERE name = 'Ravi';
-- UPDATE customers SET wallet = 0    WHERE name = 'Priya';
--
-- INSERT INTO orders (customer_id, amount, status) VALUES
--                                                      (1, 250, 'PENDING'),
--                                                      (1, 400, 'PENDING'),
--                                                      (3, 150, 'PENDING');

-- create or replace procedure cancel_all_pending(
--     IN p_customer_id NUMERIC
-- )
-- language plpgsql
-- as $$
--     DECLARE
--         v_customer_name TEXT;
--         v_cancelled_count INT := 0;
--         rec RECORD;
--     BEGIN
--         SELECT name INTO v_customer_name from customers
--         where id = p_customer_id;
--         IF v_customer_name is NULL THEN
--             RAISE EXCEPTION 'Customer % not found',p_customer_id;
--         end if;
--         FOR rec IN select * from orders where customer_id = p_customer_id LOOP
--             IF rec.status = 'PENDING' THEN
--                 UPDATE orders set STATUS = 'CANCELLED' where id = rec.id;
--                 v_cancelled_count = v_cancelled_count + 1;
--             end if;
--         end loop;
--         IF v_cancelled_count = 0 THEN
--             RAISE NOTICE 'No pending orders for customer %',p_customer_id;
--         ELSE
--             INSERT INTO audit_log (message) values('Cancelled ' || v_cancelled_count || 'orders for customer ' || p_customer_id);
--         end if;
--     END;
-- $$;
--
--
--
-- -- CALL cancel_all_pending(1);   -- Sai: 2 pending → 'Cancelled 2 orders for customer 1'
-- -- CALL cancel_all_pending(1);   -- again! → 0 pending now → NOTICE, no audit row
-- CALL cancel_all_pending(2);
--
--
-- CALL cancel_all_pending(99);
--
--
-- SELECT * FROM audit_log;



-- CREATE OR REPLACE PROCEDURE crash_test(IN p_customer_id INT)
--     LANGUAGE plpgsql
-- AS $$
-- BEGIN
--     UPDATE customers SET wallet = wallet + 99999 WHERE id = p_customer_id;  -- step 1: get rich
--     RAISE NOTICE 'wallet updated! now crashing...';
--     PERFORM 1/0;    -- step 2: 💥 divide by zero (PERFORM = run and discard result)
-- END;
-- $$;
--
-- CALL crash_test(1);
-- SELECT wallet FROM customers WHERE name = 'Sai';   -- prediction time, macha...

-- CREATE OR REPLACE PROCEDURE crash_test_caught(IN p_customer_id INT)
--     LANGUAGE plpgsql
-- AS $$
-- BEGIN
--     UPDATE customers SET wallet = wallet + 99999 WHERE id = p_customer_id;
--     RAISE NOTICE 'wallet updated! now crashing...';
--     PERFORM 1/0;                                   -- 💥
-- EXCEPTION
--     WHEN OTHERS THEN                               -- the catch block
--         RAISE NOTICE 'caught it! error was: %', SQLERRM;
-- END;
-- $$;

-- CALL crash_test_caught(1);
-- SELECT wallet FROM customers WHERE name = 'Sai';



-- -- CONSOLE 1:
-- BEGIN;
-- UPDATE customers SET wallet = wallet + 1000 WHERE name = 'Sai';
-- SELECT wallet FROM customers WHERE name = 'Sai';    -- sees 6500 (its own draft)

-- -- CONSOLE 2 (don't touch console 1):
-- SELECT wallet FROM customers WHERE name = 'Sai';    -- ❓ prediction: 6500 or 5500?

-- -- CONSOLE 1:
-- COMMIT;

-- -- CONSOLE 2 again:
-- SELECT wallet FROM customers WHERE name = 'Sai';    -- ❓ now what?